In [ ]:
import time
from pathlib import Path

import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString


# ============================================================
# 1. PATH
# ============================================================

OUTDIR = Path(
    r"..\Output"
)

INPUT_FILE = OUTDIR / "01_Hasil_Euclidean_Desa_FKTP.csv"

OUTPUT_CSV = OUTDIR / "03_Hasil_Network_OSRM_FINAL.csv"
OUTPUT_XLSX = OUTDIR / "03_Hasil_Network_OSRM_FINAL.xlsx"

TEMP_FILE = OUTDIR / "03_OSRM_TEMP.csv"

ROUTE_FILE = OUTDIR / "03_Route_Desa_FKTP.gpkg"


# ============================================================
# 2. OSRM SERVER
# ============================================================

BASE_URL = (
    "https://router.project-osrm.org/route/v1/driving/"
)


# ============================================================
# 3. REQUEST ROUTE FROM OSRM
# ============================================================

def get_route(lon1, lat1, lon2, lat2):

    url = (
        f"{BASE_URL}"
        f"{lon1},{lat1};"
        f"{lon2},{lat2}"
        f"?overview=full"
        f"&geometries=geojson"
    )

    response = requests.get(
        url,
        timeout=60
    )

    response.raise_for_status()

    return response.json()


# ============================================================
# 4. CACHE KEY
# ============================================================

def create_cache_key(row):

    return (
        str(row["KDDESA"]),
        str(row["kode_ppk"]),

        round(float(row["Longitude_Desa"]), 5),
        round(float(row["Latitude_Desa"]), 5),

        round(float(row["Longitude_FKTP"]), 5),
        round(float(row["Latitude_FKTP"]), 5)
    )


# ============================================================
# 5. MAIN PROCESS
# ============================================================

def main():

    print("=" * 70)
    print("SCRIPT 02 - NETWORK DISTANCE DESA -> FKTP")
    print("OSRM PUBLIC ROUTING")
    print("=" * 70)

    print()
    print("Membaca data Euclidean...")

    hasil = pd.read_csv(
        INPUT_FILE,
        dtype={
            "KDDESA": str,
            "kode_ppk": str
        }
    )

    print(
        f"Jumlah Desa: {len(hasil)}"
    )

    # ========================================================
    # VALIDASI KOORDINAT
    # ========================================================

    coordinate_columns = [
        "Longitude_Desa",
        "Latitude_Desa",
        "Longitude_FKTP",
        "Latitude_FKTP"
    ]

    for col in coordinate_columns:

        hasil[col] = pd.to_numeric(
            hasil[col],
            errors="coerce"
        )

    before = len(hasil)

    hasil = hasil.dropna(
        subset=coordinate_columns
    ).reset_index(drop=True)

    print(
        f"Baris tanpa koordinat valid dibuang: "
        f"{before - len(hasil)}"
    )

    print(
        f"Data yang akan diproses: {len(hasil)}"
    )

    # ========================================================
    # CACHE HASIL LAMA
    # ========================================================

    cache = {}

    if OUTPUT_CSV.exists():

        print()
        print("Membaca hasil OSRM sebelumnya...")

        previous = pd.read_csv(
            OUTPUT_CSV,
            dtype={
                "KDDESA": str,
                "kode_ppk": str
            }
        )

        for _, row in previous.iterrows():

            if row.get("Status") in [
                "ACCESS",
                "NO ROAD ACCESS"
            ]:

                try:

                    key = create_cache_key(row)

                    cache[key] = row.to_dict()

                except Exception:

                    pass

        print(
            f"Cache hasil lama: {len(cache)}"
        )

    # ========================================================
    # PROCESS ROUTING
    # ========================================================

    records = []

    routes = []

    request_count = 0

    total = len(hasil)

    for position, (_, row) in enumerate(
        hasil.iterrows(),
        start=1
    ):

        key = create_cache_key(row)

        # ----------------------------------------------------
        # PAKAI CACHE
        # ----------------------------------------------------

        if key in cache:

            records.append(
                cache[key]
            )

            print(
                f"{position}/{total} "
                f"{row['NMDESA2']} "
                f"-> CACHE"
            )

            continue

        # ----------------------------------------------------
        # REQUEST BARU
        # ----------------------------------------------------

        request_count += 1

        print(
            f"{position}/{total} "
            f"{row['NMDESA2']} "
            f"-> REQUEST OSRM"
        )

        record = row.to_dict()

        success = False

        for attempt in range(1, 4):

            try:

                result = get_route(
                    row["Longitude_Desa"],
                    row["Latitude_Desa"],

                    row["Longitude_FKTP"],
                    row["Latitude_FKTP"]
                )

                # ====================================================
                # ROUTE FOUND
                # ====================================================

                if result.get("code") == "Ok":

                    route_data = result["routes"][0]

                    network_meter = route_data["distance"]

                    network_km = (
                        network_meter / 1000
                    )

                    duration_minute = (
                        route_data["duration"] / 60
                    )

                    record["Network_Meter"] = round(
                        network_meter,
                        2
                    )

                    record["Network_KM"] = round(
                        network_km,
                        3
                    )

                    record["Duration_Minute"] = round(
                        duration_minute,
                        2
                    )

                    record["Status"] = "ACCESS"

                    # ------------------------------------------------
                    # SIMPAN GEOMETRI ROUTE
                    # ------------------------------------------------

                    route_geometry = LineString(
                        route_data[
                            "geometry"
                        ][
                            "coordinates"
                        ]
                    )

                    routes.append({

                        "KDDESA": row["KDDESA"],

                        "NMDESA2": row["NMDESA2"],

                        "kode_ppk": row["kode_ppk"],

                        "nmppk": row["nmppk"],

                        "Network_KM": record[
                            "Network_KM"
                        ],

                        "Duration_Minute": record[
                            "Duration_Minute"
                        ],

                        "geometry": route_geometry

                    })

                # ====================================================
                # NO ROUTE FOUND
                # ====================================================

                else:

                    record["Network_Meter"] = None

                    record["Network_KM"] = None

                    record["Duration_Minute"] = None

                    record["Status"] = (
                        "NO ROAD ACCESS"
                    )

                success = True

                break

            except Exception as error:

                print(
                    f"   Retry {attempt}/3"
                )

                print(
                    f"   Error: {error}"
                )

                time.sleep(3)

        # ========================================================
        # SERVER ERROR
        # ========================================================

        if not success:

            record["Network_Meter"] = None

            record["Network_KM"] = None

            record["Duration_Minute"] = None

            record["Status"] = (
                "SERVER ERROR"
            )

        records.append(
            record
        )

        # ========================================================
        # AUTO SAVE
        # ========================================================

        if position % 10 == 0:

            temporary_output = pd.DataFrame(
                records
            )

            temporary_output.to_csv(
                TEMP_FILE,
                index=False,
                encoding="utf-8"
            )

            print()
            print(
                "AUTO SAVE:"
            )

            print(
                TEMP_FILE
            )

        # ========================================================
        # DELAY API
        # ========================================================

        time.sleep(1)

    # ============================================================
    # DATAFRAME FINAL
    # ============================================================

    output = pd.DataFrame(
        records
    )

    # ============================================================
    # ROUTE GEOMETRY
    # ============================================================

    new_routes = None

    if routes:

        new_routes = gpd.GeoDataFrame(
            routes,
            geometry="geometry",
            crs="EPSG:4326"
        )

    # ------------------------------------------------------------
    # GABUNG ROUTE LAMA
    # ------------------------------------------------------------

    if ROUTE_FILE.exists():

        print()
        print(
            "Membaca route geometry lama..."
        )

        old_routes = gpd.read_file(
            ROUTE_FILE
        )

        if new_routes is not None:

            old_routes = old_routes[
                ~old_routes[
                    "KDDESA"
                ].astype(str).isin(
                    new_routes[
                        "KDDESA"
                    ].astype(str)
                )
            ]

            combined_routes = pd.concat(
                [
                    old_routes,
                    new_routes
                ],
                ignore_index=True
            )

            combined_routes = gpd.GeoDataFrame(
                combined_routes,
                geometry="geometry",
                crs="EPSG:4326"
            )

        else:

            combined_routes = old_routes

    else:

        combined_routes = new_routes

    # ------------------------------------------------------------
    # SIMPAN ROUTE
    # ------------------------------------------------------------

    if (
        combined_routes is not None
        and len(combined_routes) > 0
    ):

        combined_routes = combined_routes[
            combined_routes[
                "KDDESA"
            ].astype(str).isin(
                output[
                    "KDDESA"
                ].astype(str)
            )
        ]

        combined_routes.to_file(
            ROUTE_FILE,
            layer="Route_Desa_FKTP",
            driver="GPKG"
        )

    # ============================================================
    # SIMPAN HASIL FINAL
    # ============================================================

    output.to_csv(
        OUTPUT_CSV,
        index=False,
        encoding="utf-8"
    )

    output.to_excel(
        OUTPUT_XLSX,
        index=False
    )

    # ============================================================
    # SUMMARY
    # ============================================================

    print()
    print("=" * 70)
    print("SELESAI")
    print("=" * 70)

    print(
        f"Request OSRM baru : {request_count}"
    )

    print()

    for status in [
        "ACCESS",
        "NO ROAD ACCESS",
        "SERVER ERROR"
    ]:

        print(
            f"{status:20}: "
            f"{(
                output["Status"] == status
            ).sum()}"
        )

    print()

    if combined_routes is not None:

        print(
            f"Route geometry     : "
            f"{len(combined_routes)}"
        )

    print()

    print(
        "Output CSV:"
    )

    print(
        OUTPUT_CSV
    )

    print()

    print(
        "Output Excel:"
    )

    print(
        OUTPUT_XLSX
    )

    print()

    print(
        "Output Route:"
    )

    print(
        ROUTE_FILE
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()

In [ ]:
import pandas as pd

# ============================================================
# ADD TAPA MANUALLY AS NO ROAD ACCESS
# ============================================================

OUTPUT_DIR = r"..\Output"

CSV_FILE = f"{OUTPUT_DIR}\\03_Hasil_Network_OSRM_FINAL.csv"
EXCEL_FILE = f"{OUTPUT_DIR}\\03_Hasil_Network_OSRM_FINAL.xlsx"

# ------------------------------------------------------------
# 1. Read the original Euclidean data
# ------------------------------------------------------------

euclidean_file = f"{OUTPUT_DIR}\\01_Hasil_Euclidean_Desa_FKTP.csv"

euclidean = pd.read_csv(
    euclidean_file,
    dtype={"KDDESA": str, "kode_ppk": str}
)

# ------------------------------------------------------------
# 2. Find Tapa
# ------------------------------------------------------------

tapa = euclidean[
    euclidean["NMDESA2"]
    .astype(str)
    .str.strip()
    .str.upper()
    == "TAPA"
].copy()

if len(tapa) == 0:
    print("TAPA tidak ditemukan di file Euclidean.")

else:

    # --------------------------------------------------------
    # 3. Add network result manually
    # --------------------------------------------------------

    tapa["Network_Meter"] = pd.NA
    tapa["Network_KM"] = pd.NA
    tapa["Duration_Minute"] = pd.NA
    tapa["Status"] = "NO ROAD ACCESS"

    # --------------------------------------------------------
    # 4. Read existing OSRM results
    # --------------------------------------------------------

    hasil = pd.read_csv(
        CSV_FILE,
        dtype={"KDDESA": str, "kode_ppk": str}
    )

    # Prevent duplicate Tapa if the script has already added it
    hasil = hasil[
        ~(
            hasil["NMDESA2"]
            .astype(str)
            .str.strip()
            .str.upper()
            == "TAPA"
        )
    ]

    # --------------------------------------------------------
    # 5. Append Tapa
    # --------------------------------------------------------

    hasil_final = pd.concat(
        [hasil, tapa],
        ignore_index=True
    )

    # --------------------------------------------------------
    # 6. Save final CSV
    # --------------------------------------------------------

    hasil_final.to_csv(
        CSV_FILE,
        index=False,
        encoding="utf-8"
    )

    # --------------------------------------------------------
    # 7. Save final Excel
    # --------------------------------------------------------

    hasil_final.to_excel(
        EXCEL_FILE,
        index=False
    )

    # --------------------------------------------------------
    # 8. Summary
    # --------------------------------------------------------

    print("=" * 60)
    print("TAPA BERHASIL DITAMBAHKAN")
    print("=" * 60)

    print(f"Total desa          : {len(hasil_final)}")
    print(
        f"ACCESS              : "
        f"{(hasil_final['Status'] == 'ACCESS').sum()}"
    )
    print(
        f"NO ROAD ACCESS      : "
        f"{(hasil_final['Status'] == 'NO ROAD ACCESS').sum()}"
    )

    print("\nData Tapa:")
    print(
        hasil_final[
            hasil_final["NMDESA2"]
            .astype(str)
            .str.strip()
            .str.upper()
            == "TAPA"
        ].to_string(index=False)
    )

    print("\nFile berhasil diperbarui:")
    print(CSV_FILE)
    print(EXCEL_FILE)